# 🦙 LlamaIndex — সম্পূর্ণ Reference Notebook
## এক নোটবুকে সব কিছু!

| ধাপ | বিষয় | গুরুত্বপূর্ণ কোড |
|-----|-------|------------------|
| Step 1 | Document Loading | `SimpleDirectoryReader`, `Document` |
| Step 2 | Model Settings | `Settings.embed_model`, `Settings.llm` |
| Step 3 | Text Splitting | `SentenceSplitter`, `SemanticSplitter` |
| Step 4 | Vector Store (Chroma) | `ChromaVectorStore`, `VectorStoreIndex` |
| Step 5 | Query Engine | `as_query_engine()`, `.query()` |
| Step 6 | Chat Engine | `as_chat_engine()`, `.chat()` |
| Step 7 | Retriever | `as_retriever()`, `MetadataFilters` |
| Step 8 | Custom Prompt | `PromptTemplate`, `text_qa_template` |
| Step 9 | Index Persistence | `.persist()`, `load_index_from_storage()` |
| Step 10 | Ingestion Pipeline | `IngestionPipeline`, `IngestionCache` |
| Step 11 | Custom Transform | `TransformComponent` |
| Step 12 | Evaluation | `FaithfulnessEvaluator`, `BatchEvalRunner` |
| Step 13 | Full Production Pipeline | সব একসাথে |

---
## 📌 Step 1: Document Loading — ডকুমেন্ট লোড করা

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Document

# ── ১. ফোল্ডার থেকে লোড (সব ফাইল) ──
documents = SimpleDirectoryReader(input_dir="Data").load_data()
print(f"📄 {len(documents)}টি document লোড হয়েছে")

# ── ২. শুধু নির্দিষ্ট extension ──
pdf_docs = SimpleDirectoryReader(
    input_dir="Data",
    required_exts=[".pdf"]
).load_data()

# ── ৩. Subfolder সহ (recursive) ──
all_docs = SimpleDirectoryReader(
    input_dir="Data",
    recursive=True
).load_data()

# ── ৪. একটা নির্দিষ্ট ফাইল ──
single = SimpleDirectoryReader(
    input_files=["Data/practice_llamaindex.txt"]
).load_data()

# ── ৫. Custom Metadata সহ ──
meta_docs = SimpleDirectoryReader(
    input_dir="Data",
    file_metadata=lambda path: {"source": "my_project", "path": path}
).load_data()

# ── ৬. Manual Document তৈরি ──
manual_doc = Document(
    text="LlamaIndex is a RAG framework.",
    metadata={"source": "manual", "author": "Maruf"}
)

# ── Document Metadata দেখা ──
print("\n📋 Document Info:")
print(f"  File: {documents[0].metadata.get('file_name')}")
print(f"  Text (100 char): {documents[0].text[:100]}")

---
## 📌 Step 2: Model Settings — LLM ও Embedding সেট করা

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM
from dotenv import load_dotenv
load_dotenv()

# ── Embedding Model ──
# Option A: HuggingFace (ফ্রি)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # 384 dim
)

# Option B: OpenAI Embedding
# from llama_index.embeddings.openai import OpenAIEmbedding
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# ── LLM ──
# Option A: MockLLM (test-এর জন্য, real উত্তর দেয় না)
Settings.llm = MockLLM(max_tokens=256)

# Option B: OpenAI (real উত্তর দেয়, API key লাগে)
# from llama_index.llms.openai import OpenAI
# Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

print(f"✅ Embed Model: {Settings.embed_model.model_name}")
print(f"✅ LLM: {type(Settings.llm).__name__}")

---
## 📌 Step 3: Text Splitting — ডকুমেন্ট ভাগ করা (Node তৈরি)

In [ ]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    TokenTextSplitter,
    SemanticSplitterNodeParser,
    MarkdownElementNodeParser,
    HTMLNodeParser,
    CodeSplitter,
)

# ── ১. SentenceSplitter (সবচেয়ে common) ──
sentence_splitter = SentenceSplitter(
    chunk_size=256,    # সর্বোচ্চ ২৫৬ character
    chunk_overlap=50   # পরের chunk-এ ৫০ char আগের chunk থেকে
)

# ── ২. Token-based Splitter ──
token_splitter = TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

# ── ৩. Semantic Splitter (অর্থ বুঝে ভাগ করে) ──
semantic_splitter = SemanticSplitterNodeParser(
    embed_model=Settings.embed_model,
    buffer_size=1,
    breakpoint_percentile_threshold=95
)

# ── ৪. Markdown Splitter ──
md_splitter = MarkdownElementNodeParser()

# ── ৫. HTML Splitter ──
html_splitter = HTMLNodeParser()

# ── ৬. Code Splitter ──
code_splitter = CodeSplitter(
    language="python",
    chunk_lines=40,
    chunk_lines_overlap=5
)

# ── Documents থেকে Nodes তৈরি ──
nodes = sentence_splitter.get_nodes_from_documents(documents)
print(f"✅ {len(documents)} documents → {len(nodes)} nodes")
print(f"   Node 1 text: {nodes[0].text[:80]}...")

---
## 📌 Step 4: Vector Store — Index তৈরি ও ChromaDB

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

# ── ১. Simple In-Memory Index (সবচেয়ে সহজ) ──
simple_index = VectorStoreIndex.from_documents(
    documents=documents,
    transformations=[sentence_splitter]
)
print("✅ Simple In-Memory Index তৈরি!")

# ── ২. ChromaDB সহ Index (Persistent) ──
chroma_client = chromadb.Client()  # In-memory
# Disk-এ save করতে:
# chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection   = chroma_client.get_or_create_collection("my_collection")
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_ctx  = StorageContext.from_defaults(vector_store=vector_store)

chroma_index = VectorStoreIndex.from_documents(
    documents=documents,
    storage_context=storage_ctx,
    transformations=[sentence_splitter]
)
print(f"✅ Chroma Index তৈরি! Items: {collection.count()}")

# ── Chroma Collection CRUD ──
print(f"   Count: {collection.count()}")
# collection.get(limit=3)         # ডেটা দেখা
# collection.delete(ids=["id1"]) # ডিলিট করা

# ── ৩. Nodes থেকে সরাসরি Index ──
from llama_index.core.schema import TextNode
nodes_index = VectorStoreIndex(nodes=nodes)

---
## 📌 Step 5: Query Engine — প্রশ্ন করা

In [ ]:
# ── ১. Basic Query Engine ──
query_engine = simple_index.as_query_engine()
response = query_engine.query("What is LlamaIndex?")
print(f"🤖 উত্তর: {response}")

# ── ২. Response বিস্তারিত দেখা ──
print(f"\n📝 response.response: {response.response}")
print(f"📂 Source nodes: {len(response.source_nodes)}টি")
for node in response.source_nodes:
    print(f"   Score: {node.score:.4f} | File: {node.metadata.get('file_name')} | Text: {node.text[:50]}...")

# ── ৩. Parameters কাস্টমাইজ ──
custom_qe = simple_index.as_query_engine(
    similarity_top_k=3,           # কতটি chunk থেকে উত্তর খুঁজবে
    response_mode="compact",       # compact | tree_summarize | no_text
)

# ── ৪. Streaming ──
stream_qe = simple_index.as_query_engine(streaming=True)
stream_res = stream_qe.query("What is RAG?")
# stream_res.print_response_stream()

---
## 📌 Step 6: Custom Prompt ও Chat Engine

In [ ]:
from llama_index.core import PromptTemplate

# ── ১. Custom Prompt Template ──
my_prompt = PromptTemplate(
    """
    তুমি একজন AI assistant।
    Context: {context_str}
    প্রশ্ন: {query_str}
    উত্তর (বাংলায় দাও):
    """
)

prompt_qe = simple_index.as_query_engine(
    text_qa_template=my_prompt,
    similarity_top_k=2
)
print("✅ Custom Prompt সহ Query Engine তৈরি!")

# ── ২. Chat Engine (Memory সহ) ──
chat_engine = simple_index.as_chat_engine(
    chat_mode="condense_plus_context",
    verbose=False
)

r1 = chat_engine.chat("What is LlamaIndex?")
print(f"\n👤 Q1: What is LlamaIndex?")
print(f"🤖 A1: {r1}")

r2 = chat_engine.chat("What are its key components?")  # 'its' = LlamaIndex — memory!
print(f"\n👤 Q2: What are its key components?")
print(f"🤖 A2: {r2}")

# Chat history দেখা ও reset
print(f"\n📜 Chat turns: {len(chat_engine.chat_history)}")
chat_engine.reset()  # memory মুছে ফেলা
print("🔄 Chat reset!")

---
## 📌 Step 7: Retriever ও Metadata Filtering

In [ ]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

# ── ১. Basic Retriever ──
retriever = simple_index.as_retriever(similarity_top_k=3)
nodes = retriever.retrieve("What is RAG?")
print(f"📥 {len(nodes)}টি relevant chunk পাওয়া গেছে:")
for node in nodes:
    print(f"  Score: {node.score:.4f} | {node.metadata.get('file_name')} | {node.text[:60]}...")

# ── ২. Metadata Filter সহ Retriever ──
filters = MetadataFilters(
    filters=[
        MetadataFilter(key="file_name", value="practice_llamaindex.txt")
    ]
)
filtered_retriever = simple_index.as_retriever(
    similarity_top_k=3,
    filters=filters
)
filtered_nodes = filtered_retriever.retrieve("data loading")
print(f"\n🔍 Filtered: {len(filtered_nodes)} nodes (শুধু .txt ফাইল থেকে)")

---
## 📌 Step 8: Index Persistence — Disk-এ Save ও Load

In [ ]:
import os
from llama_index.core import StorageContext, load_index_from_storage

PERSIST_DIR = "./storage"

# ── Index Save করা ──
simple_index.storage_context.persist(persist_dir=PERSIST_DIR)
print(f"💾 Index saved → '{PERSIST_DIR}'")
print(f"   Files: {os.listdir(PERSIST_DIR)}")

# ── Index Load করা ──
storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
loaded_index    = load_index_from_storage(storage_context)
print(f"\n✅ Index loaded! Type: {type(loaded_index).__name__}")

# Load করা index দিয়ে query করা
loaded_qe = loaded_index.as_query_engine()
res = loaded_qe.query("What is LlamaIndex?")
print(f"🤖 Loaded index থেকে উত্তর: {res}")

---
## 📌 Step 9: Ingestion Pipeline

In [ ]:
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.schema import TransformComponent, BaseNode
from typing import Any, List

# ── ১. Custom Transformation ──
class MetadataEnricher(TransformComponent):
    author: str = "Maruf"
    def __call__(self, nodes: List[BaseNode], **kwargs: Any) -> List[BaseNode]:
        for node in nodes:
            node.metadata["author"]   = self.author
            node.metadata["char_len"] = str(len(node.text))
        return nodes

# ── ২. Pipeline তৈরি ──
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),  # Split
        MetadataEnricher(author="Maruf"),                    # Custom metadata
        Settings.embed_model,                                # Embed
    ],
    cache=IngestionCache(),           # Cache — duplicate skip করে
    docstore=SimpleDocumentStore(),   # New doc detect করে
)

# ── ৩. Pipeline চালানো ──
nodes = pipeline.run(documents=documents)
print(f"✅ Pipeline: {len(documents)} docs → {len(nodes)} nodes")
print(f"   Custom metadata: {nodes[0].metadata.get('author')}")

# ── ৪. Pipeline Save/Load ──
PIPE_DIR = "./pipeline_storage"
os.makedirs(PIPE_DIR, exist_ok=True)
pipeline.persist(persist_dir=PIPE_DIR)
print(f"💾 Pipeline cache saved → '{PIPE_DIR}'")

# Load করা:
new_pipeline = IngestionPipeline(
    transformations=[SentenceSplitter(), Settings.embed_model]
)
new_pipeline.load(persist_dir=PIPE_DIR)
print("✅ Pipeline cache loaded!")

# ── ৫. ChromaDB সহ Pipeline ──
pipeline_chroma = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ],
    vector_store=vector_store   # সরাসরি Chroma-তে save
)
chroma_nodes = pipeline_chroma.run(documents=documents)
print(f"✅ Chroma Pipeline: {len(chroma_nodes)} nodes")

---
## 📌 Step 10: Evaluation — RAG Quality মাপা

In [ ]:
# ⚠️ Evaluation-এ real LLM লাগে (OpenAI key দরকার)
# Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

from llama_index.core.evaluation import (
    FaithfulnessEvaluator,    # Hallucination detect
    RelevancyEvaluator,       # উত্তর প্রাসঙ্গিক কিনা
    CorrectnessEvaluator,     # Score 1-5 (reference লাগে)
    AnswerRelevancyEvaluator, # প্রশ্নের সরাসরি জবাব কিনা
    ContextRelevancyEvaluator,# Retrieved context ভালো কিনা
    BatchEvalRunner,          # অনেক প্রশ্ন একসাথে
    RetrieverEvaluator,       # Retriever performance
    generate_question_context_pairs,  # Auto-generate QA dataset
)

# ── ১. Faithfulness (Hallucination check) ──
faith_eval = FaithfulnessEvaluator(llm=Settings.llm)
response   = query_engine.query("What is LlamaIndex?")
# result = faith_eval.evaluate_response(response=response)
# print(f"Faithful: {result.passing} | Score: {result.score}")

# ── ২. Relevancy ──
relev_eval = RelevancyEvaluator(llm=Settings.llm)
# result = relev_eval.evaluate_response(query="What is RAG?", response=response)

# ── ৩. Correctness (reference answer দিতে হবে) ──
correct_eval = CorrectnessEvaluator(llm=Settings.llm)
# result = correct_eval.evaluate_response(
#     query="What is LlamaIndex?",
#     response=response,
#     reference="LlamaIndex is a data framework for LLM apps."
# )
# print(f"Score (1-5): {result.score} | Passing: {result.passing}")

# ── ৪. Batch Evaluation ──
# runner = BatchEvalRunner(
#     evaluators={"faithfulness": faith_eval, "relevancy": relev_eval},
#     workers=2
# )
# eval_results = await runner.aevaluate_queries(
#     query_engine,
#     queries=["What is LlamaIndex?", "What is RAG?"]
# )

# ── ৫. Auto QA Dataset Generate ──
# qa_dataset = generate_question_context_pairs(
#     nodes=nodes, llm=Settings.llm, num_questions_per_chunk=2
# )
# qa_dataset.save_json("qa_dataset.json")

# ── ৬. Retriever Evaluation ──
# retriever_eval = RetrieverEvaluator.from_metric_names(
#     metric_names=["hit_rate", "mrr"], retriever=retriever
# )
# results = await retriever_eval.aevaluate_dataset(qa_dataset)

print("✅ Evaluation classes import সম্পন্ন!")
print("   (OpenAI key থাকলে comment (#) সরিয়ে run করো)")

---
## 📌 Step 11: পুরো Production Pipeline — সব একসাথে
Real project-এ যেভাবে ব্যবহার করা হয়।

In [ ]:
import os
import chromadb
from llama_index.core import (
    Settings, SimpleDirectoryReader,
    VectorStoreIndex, StorageContext,
    load_index_from_storage, PromptTemplate
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM
from dotenv import load_dotenv
load_dotenv()

# ════════════════════════════════════════
#  CONFIG
# ════════════════════════════════════════
DATA_DIR     = "Data"
CHROMA_DIR   = "./prod_chroma"
PIPELINE_DIR = "./prod_pipeline"
COLLECTION   = "prod_collection"

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
Settings.llm = MockLLM(max_tokens=256)
# Real LLM: Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

# ════════════════════════════════════════
#  STORAGE — ChromaDB
# ════════════════════════════════════════
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection    = chroma_client.get_or_create_collection(COLLECTION)
vector_store  = ChromaVectorStore(chroma_collection=collection)
storage_ctx   = StorageContext.from_defaults(vector_store=vector_store)
print(f"✅ ChromaDB ready | Items: {collection.count()}")

# ════════════════════════════════════════
#  PIPELINE — Load or Create
# ════════════════════════════════════════
os.makedirs(PIPELINE_DIR, exist_ok=True)
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=50),
        Settings.embed_model,
    ],
    vector_store=vector_store,
    cache=IngestionCache(),
    docstore=SimpleDocumentStore(),
)
if os.path.exists(os.path.join(PIPELINE_DIR, "docstore.json")):
    pipeline.load(persist_dir=PIPELINE_DIR)
    print("📂 Pipeline cache loaded!")

# Documents process করা
documents = SimpleDirectoryReader(input_dir=DATA_DIR).load_data()
nodes     = pipeline.run(documents=documents)
pipeline.persist(persist_dir=PIPELINE_DIR)
print(f"✅ Pipeline: {len(nodes)} nodes | Chroma: {collection.count()} items")

# ════════════════════════════════════════
#  INDEX ও QUERY ENGINE
# ════════════════════════════════════════
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_ctx
)

# Custom Prompt
my_prompt = PromptTemplate(
    "Context: {context_str}\n\nQuestion: {query_str}\nAnswer concisely:"
)

query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact",
    text_qa_template=my_prompt
)

# Chat Engine
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context"
)

# ════════════════════════════════════════
#  QUERY করো!
# ════════════════════════════════════════
print("\n" + "═"*55)
questions = ["What is LlamaIndex?", "What is RAG?", "What is chunking?"]
for q in questions:
    res = query_engine.query(q)
    srcs = [n.metadata.get('file_name','?') for n in res.source_nodes]
    print(f"\n❓ {q}")
    print(f"🤖 {res.response}")
    print(f"📂 Sources: {srcs}")
print("═"*55)

---
## 🗒️ Quick Cheatsheet — সব গুরুত্বপূর্ণ কোড এক পাতায়

```python
# ═══════════════════════════════════════════════════════
#  IMPORT
# ═══════════════════════════════════════════════════════
from llama_index.core import (
    Settings, SimpleDirectoryReader, Document,
    VectorStoreIndex, StorageContext,
    load_index_from_storage, PromptTemplate
)
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# ═══════════════════════════════════════════════════════
#  SETTINGS
# ═══════════════════════════════════════════════════════
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

# ═══════════════════════════════════════════════════════
#  LOAD
# ═══════════════════════════════════════════════════════
docs = SimpleDirectoryReader(input_dir="Data").load_data()
docs = SimpleDirectoryReader(input_dir="Data", required_exts=[".pdf"]).load_data()
docs = SimpleDirectoryReader(input_dir="Data", recursive=True).load_data()
doc  = Document(text="...", metadata={"source": "manual"})

# ═══════════════════════════════════════════════════════
#  SPLIT
# ═══════════════════════════════════════════════════════
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=50)
nodes    = splitter.get_nodes_from_documents(docs)

# ═══════════════════════════════════════════════════════
#  INDEX
# ═══════════════════════════════════════════════════════
index = VectorStoreIndex.from_documents(docs, transformations=[splitter])
# Save:
index.storage_context.persist(persist_dir="./storage")
# Load:
index = load_index_from_storage(StorageContext.from_defaults(persist_dir="./storage"))

# ═══════════════════════════════════════════════════════
#  QUERY ENGINE
# ═══════════════════════════════════════════════════════
qe  = index.as_query_engine(similarity_top_k=3, response_mode="compact")
res = qe.query("প্রশ্ন")
print(res.response)         # উত্তর
print(res.source_nodes)     # কোথা থেকে এসেছে

# ═══════════════════════════════════════════════════════
#  CHAT ENGINE
# ═══════════════════════════════════════════════════════
ce  = index.as_chat_engine(chat_mode="condense_plus_context")
res = ce.chat("Hello!")
ce.reset()                  # memory মুছে ফেলা

# ═══════════════════════════════════════════════════════
#  RETRIEVER
# ═══════════════════════════════════════════════════════
ret   = index.as_retriever(similarity_top_k=3)
nodes = ret.retrieve("প্রশ্ন")
# Filter:
ret = index.as_retriever(filters=MetadataFilters(filters=[MetadataFilter(key="file_name", value="abc.txt")]))

# ═══════════════════════════════════════════════════════
#  PIPELINE
# ═══════════════════════════════════════════════════════
pipeline = IngestionPipeline(
    transformations=[splitter, Settings.embed_model],
    vector_store=vector_store,
    cache=IngestionCache(),
    docstore=SimpleDocumentStore()
)
nodes = pipeline.run(documents=docs)
pipeline.persist(persist_dir="./pipeline")

# ═══════════════════════════════════════════════════════
#  EVALUATION (OpenAI key দরকার)
# ═══════════════════════════════════════════════════════
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator, CorrectnessEvaluator, BatchEvalRunner
result = FaithfulnessEvaluator(llm=Settings.llm).evaluate_response(response=res)
print(result.passing, result.score, result.feedback)
```

---
## 📊 তোমার এখন পর্যন্ত LlamaIndex Progress

| Topic | Status |
|-------|--------|
| Document Loading | ✅ Done |
| Model Settings | ✅ Done |
| Text Splitting | ✅ Done |
| Vector Store (Chroma) | ✅ Done |
| Query Engine | ✅ Done |
| Chat Engine | ✅ Done |
| Retriever & Filtering | ✅ Done |
| Index Persistence | ✅ Done |
| Ingestion Pipeline | ✅ Done |
| Custom Transform | ✅ Done |
| Evaluation | ✅ Done |
| **Agents & Tools** | 🔜 Next |
| **Advanced Retrieval** | 🔜 Next |
| **Router Query Engine** | 🔜 Next |